In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("ALPACA_API_KEY")
secret_key = os.getenv("ALPACA_SECRET_KEY")

print("Keys cargadas:", api_key is not None, secret_key is not None)

In [ ]:
!pip install alpaca-py

In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime

In [ ]:
client = StockHistoricalDataClient(api_key, secret_key)

In [ ]:
request_params = StockBarsRequest(
    symbol_or_symbols="TSLA",
    timeframe=TimeFrame.Day,
    start=datetime(2020, 1, 1),
    end=datetime(2026, 7, 21)
)

bars = client.get_stock_bars(request_params)
df = bars.df

print(df.shape)
df.head()

In [ ]:
df.to_csv("../data/raw/tsla_raw.csv")
print("Guardado correctamente")

In [ ]:
df = df.sort_index()

df['retorno_diario'] = df['close'].pct_change()
df['sma_5'] = df['close'].rolling(window=5).mean()
df['sma_20'] = df['close'].rolling(window=20).mean()
df['volatilidad_5'] = df['close'].rolling(window=5).std()
df['volumen_promedio_5'] = df['volume'].rolling(window=5).mean()
df['precio_anterior'] = df['close'].shift(1)
df['precio_futuro'] = df['close'].shift(-1)

print(df.shape)
df.head(10)

In [ ]:
print(df.isnull().sum())

df_limpio = df.dropna()

print("Antes:", df.shape)
print("Después de limpiar:", df_limpio.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

features = ['open', 'high', 'low', 'close', 'volume', 'retorno_diario',
            'sma_5', 'sma_20', 'volatilidad_5', 'volumen_promedio_5', 'precio_anterior']

X = df_limpio[features]
y = df_limpio['precio_futuro']

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

print("Forma de X escalado:", X_escalado.shape)
print("Forma de y:", y.shape)

In [ ]:
import pandas as pd

df_final = pd.DataFrame(X_escalado, columns=features)
df_final['precio_futuro'] = y.values

df_final.to_csv("../data/processed/tsla_processed.csv", index=False)

print("Dataset final guardado correctamente")
print(df_final.shape)
df_final.head()